In [4]:
import hail as hl
import os

In [2]:
from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [5]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')

In [6]:
hail_init.hail_bmrc_init('/logs/hail/hail_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe055.hpc.in.bmrc.ox.ac.uk:4041
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.74-0c3a74d12093
LOGGING: writing to /logs/hail/hail_export.log


In [20]:
#
mt1 = qc.get_table('data/mt/ukb_wes_200k_annotated_chr22.mt', 'mt')
mt1.vep.worst_csq_by_gene_canonical.gene_symbol
#mt1 = qc.filter_min_missing(mt1, 0.05)
#mt1 = qc.filter_max_maf(mt1, 0.02)

,,
locus,alleles,
locus<GRCh38>,array<str>,array<str>
chr22:15528182,"[""C"",""T""]","[""OR11H1""]"
chr22:15528188,"[""C"",""T""]","[""OR11H1""]"
chr22:15528194,"[""G"",""A""]","[""OR11H1""]"
chr22:15528206,"[""G"",""A""]","[""OR11H1""]"
chr22:15528267,"[""A"",""G""]","[""OR11H1""]"
chr22:15528300,"[""A"",""G""]","[""OR11H1""]"
chr22:15528306,"[""T"",""A""]","[""OR11H1""]"
chr22:15528309,"[""G"",""A""]","[""OR11H1""]"


In [8]:
in_mt = mt1

In [9]:
in_mt = in_mt.annotate_entries(rsid_entry = in_mt.rsid)
in_mt = in_mt.annotate_entries(snpid_entry = in_mt.snpid)
in_mt = in_mt.annotate_entries(af_entry = in_mt.info.AF)
in_mt = in_mt.annotate_entries(ac_entry = in_mt.info.AC)
in_mt = in_mt.annotate_entries(rsid_gt = hl.delimit([in_mt.snpid_entry,
              in_mt.rsid_entry,
              hl.str(in_mt.af_entry),
              hl.str(in_mt.ac_entry),
              in_mt.vep.consequence_category,
              in_mt.vep.most_severe_consequence,
              in_mt.vep.worst_csq_by_gene_canonical.lof[0],
              in_mt.vep.worst_csq_by_gene_canonical.lof_flags[0],
              hl.str(in_mt.dbnsfp.revel_score),
              hl.str(in_mt.dbnsfp.cadd_phred_score)], ';'))

In [18]:
in_mt.vep.Gene.gene_symbol

<ArrayExpression of type array<str>>

In [ ]:
    #in_mt = in_mt.annotate_entries(rsid_gt = hl.delimit([in_mt.rsid_entry, in_mt.vep.consequence]))
    # get all snps that are not homozygous
mt = in_mt
mt = annotate_phased_entries(mt)
mt = mt.filter_entries(~mt.GT.is_hom_var())
# create table for each strand and combine to gene
ht0 = (mt.group_rows_by(mt.vep.Gene).aggregate(s0 = hl.agg.filter(mt.a0_alt, hl.agg.collect(mt.rsid_gt))))
ht1 = (mt.group_rows_by(mt.vep.Gene).aggregate(s1 = hl.agg.filter(mt.a1_alt, hl.agg.collect(mt.rsid_gt))))
ht2 = (in_mt.group_rows_by(in_mt.vep.Gene).aggregate(hom_var = hl.agg.filter(in_mt.GT.is_hom_var(), hl.agg.collect(in_mt.rsid_gt))))
# combine entries
ht = ht0.annotate_entries(s1 = ht1[ht0.Gene, ht0.s].s1)
ht = ht.annotate_entries(hom_var = ht2[ht.Gene, ht.s].hom_var)